This notebook was inspired by [GPT token encoder and decoder](https://observablehq.com/@simonw/gpt-tokenizer) and partially uses python portings of their code

In [6]:
# --- Inititialization ---
import itables
itables.options.maxBytes = "4MB"
itables.init_notebook_mode(all_interactive=True)
from utils.tokenization_init import encoder, encode, decode, generate_spans_html

# Tokenization Exercise

Large Language Models don’t process text directly — they process **tokens**, which are pieces of text such as whole words, parts of words, or punctuation.
A **tokenizer** converts text → tokens → token IDs (numbers).
During inference, these token IDs are passed to the embedding layer — the next step in the model pipeline.

This notebook lets you **explore how tokenization works** using the **GPT-2 tokenizer**.

## Building Vocabularies

Most modern language models use **subword tokenization** methods based on **Byte Pair Encoding (BPE)** or closely related algorithms.
The core idea is to iteratively merge frequently occurring character or token pairs into new tokens, gradually building an efficient vocabulary from text data.

### 🧩 Exercise 1.1 — Building a Vocabulary

1. Go to [Byte Pair Encoding Visualizer - Interactive Algorithm Demo](https://philipmueller.dev/bpe-visualization/)
    * select the `Text` Training Data
    * press `Start Training`
    * press `Play`
    * Observe how iteratively the most frequent token pairs in the text get joined into new tokens, building a vocabulary


---

## Exploring GPT-2 Vocabulary

### Disclaimer

> This notebook uses the **GPT-2 tokenizer**, which has a **vocabulary size of 50,257 tokens**.
> Modern models (e.g., GPT-3, GPT-4, LLaMA 3, Mistral) use slightly different tokenizers and may split text differently.
> The GPT-2 tokenizer is nevertheless ideal for **educational purposes**, as it illustrates the key ideas of subword tokenization and byte-pair encoding.

## Token / Token ID Table

You can browse and search through the vocabulary table below.
Each entry shows a **token** and its **token ID**.
Note that many tokens include **leading spaces** or special characters such as `"Ġ"`, `"▁"`, or `"Ċ"` — these indicate spaces, line breaks, or special byte sequences.

### 🧩 Exercise 1.2 — Vocabulary Exploration

2. Observe how spaces become part of tokens — e.g., `"_Hello"` vs `"Hello"`.
    Look up the following tokens using the Search input (top right) of the table below:
    * `"Hello"`
    * `"juice"`
    * `"questionability"`
    * `"ability"`
    * `"ization"`

    Which appear as full tokens, which only as parts of other tokens, and which words are not available as tokens at all?

In [3]:
import json
import pandas as pd

# https://observablehq.com/@codingwithfire/gpt-3-encoder
with open("encoder.json", "r") as f:
    encoder = json.load(f)

# Sort by token length and replace "Ġ" with "_"
sorted_items = sorted(
    ((token.replace("Ġ", "▁"), token_id) for token, token_id in encoder.items()),
    key=lambda x: len(x[0])
)

vocabulary = pd.DataFrame(sorted_items, columns=["token", "token_id"])

itables.show(
    vocabulary,      
    autoWidth= False,
    columnDefs= [{"width": "50%", "targets": [0, 1]}],
    style="width:80%;margin:auto",
    search={"regex": False, "caseInsensitive": False, "search": ""}
)

Loading ITables v2.5.2 from the init_notebook_mode cell... (need help?)


## [Optional] Rebuild from Token IDs
This cell allows you to rebuild tokens from their ids by providing the token ids in the input below.

Enter a sequence of space-separated **token IDs** below.  
The notebook will decode them back into text using the tokenizer.  
Use this to check whether your selected token IDs correctly reconstruct words or emojis.

**Example:**  
`15496 2159` → “Hello World”

In [4]:
import ipywidgets as widgets
from IPython.display import display
import re, html

tokens = widgets.Text(
    value="",
    placeholder="Paste space separated token ids here",
    description="Token IDs:",
    layout=widgets.Layout(width="80%")
)

decoded_html = widgets.HTML()

def parse_token_ids(s: str):
    return [int(x) for x in re.findall(r"\d+", s)]

def render_decoded(s: str):
    # validate input format first
    if not re.fullmatch(r"(\d{1,6}( \d{1,6})*)?", s.strip()):
        decoded_html.value = (
            "<pre style='color:red'>Error: Invalid format. "
            "Use space-separated token ids only!</pre>"
        )
        return
    
    try:
        ids = parse_token_ids(s)
        txt = decode(ids) if ids else ""
        decoded_html.value = f"<pre>{html.escape(txt)}</pre>"
    except Exception as e:
        decoded_html.value = f"<pre style='color:red'>Error: {html.escape(str(e))}</pre>"

# connect updates
render_decoded(tokens.value)
tokens.observe(lambda ch: ch['name'] == 'value' and render_decoded(ch['new']), names='value')

display(tokens)
display(decoded_html)


Text(value='', description='Token IDs:', layout=Layout(width='80%'), placeholder='Paste space separated token …

HTML(value='<pre></pre>')

---

## Text → Tokens and IDs
Type any text below — the notebook will display how it is tokenized, showing **each token** and its **corresponding ID**.

### 🧩 Exercise 2 — Tokenization

1. Tokenize **“Helmholtz”**. Does it match your reconstruction above?
2. Test words ending in “ization”:  
   * `generalization`, `realization`, `neutralization`, `capitalization`  
   Compare how much of “ization” is reused between them.
3. Examine the effect of highlighting and capitalization for the word **“title”**:  
   * `Title`, `TITLE`, `**title**`, `TItLe`  
   How does formatting or casing affect tokenization?
4. LLMs use special tokens such as `<|endoftext|>` (search for it in the table!) to mark the end of generation.  
Type this token in the input field. What happens when a user attempts to put this special token in their input text?

In [7]:
import ipywidgets as widgets
from IPython.display import display
import html

text = widgets.Text(
    value="Example text is here",
    placeholder="Enter text to tokenize",
    description="Text:",
    layout=widgets.Layout(width="80%")
)

display(text)

# --------------------------------------------------------
tokens_html = widgets.HTML()

def render_tokens_and_count(value: str):
    toks = encode(value)
    joined = " ".join(str(t) for t in toks)
    count = len(toks)
    plural = "" if count == 1 else "s"
    tokens_html.value = f"<code>{joined}</code><br><small>{count} token{plural}</small>"

# Initial render + reactive updates
render_tokens_and_count(text.value)
text.observe(lambda ch: ch['name'] == 'value' and render_tokens_and_count(ch['new']), names='value')

display(tokens_html)

# --------------------------------------------------------
spans_html = widgets.HTML()

def render_spans(value: str):
    spans_html.value = generate_spans_html(value)

# Initial render + reactive updates
render_spans(text.value)
text.observe(lambda ch: ch['name'] == 'value' and render_spans(ch['new']), names='value')

display(spans_html)

Text(value='Example text is here', description='Text:', layout=Layout(width='80%'), placeholder='Enter text to…

HTML(value='<code>16281 2420 318 994</code><br><small>4 tokens</small>')

HTML(value='<div><span title="[16281]" style="\n        padding: 3px;\n        border-right: 3px solid white;\…

### Extra (optional) Exploration

1. Open the [**OpenAI Tokenizer**](https://platform.openai.com/tokenizer) for GPT-3, GPT-4, and GPT-4-mini.  
   * Compare how “Bücherregal” and “november” are split here for GPT-2 and in for GPT-3/GPT-4 in the online tool.
2. Try emojis:  
   * Tokenize “🚴”. How many tokens is it?  
   * Copy the token IDs into the “Rebuild from Token IDs” cell above — does the emoji reappear?
3. Experiment with technical terms from your own field or words from your native language (if not English).  
   * Which ones are single tokens, and which are composed of several sub-tokens?  
   * What might that imply for performance or cost?

#### Vocabulary Sizes per Model
| Model                                 | Approximate Vocabulary Size           |
| ------------------------------------- | ------------------------------------- |
| cl100k_base (used by **GPT-3.5 / GPT-4**) | ~ **100,256 tokens**      |
| o200k_base (used by **GPT-4o**)           | ~ **199,997 tokens**     |
| Earlier **GPT-2** tokenizer               | **50,257 tokens** |


### Discussion Prompts

* Why does it matter how many tokens a word has?
* How could tokenization affect model cost, context length, or even bias?